# Data Balancing Pipeline for Vishing Detection

This notebook implements a robust pipeline to counteract the dataset's class imbalance (5% original fraud), generating new datasets using various techniques:
- **Random Oversampling**
- **SMOTE** (Synthetic Minority Over-sampling Technique)
- **Borderline SMOTE**
- **SMOTE + Undersampling** (Hybrid)

We will generate versions with **10%**, **20%**, and **25%** proportions of the minority class (`is_vishing`).

## 1. Import Libraries and Configure Environment

In [1]:
import os
import pandas as pd
import numpy as np

# Imblearn - Oversampling and Hybrids
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load and Prepare Original Dataset

In [2]:
RAW_DATA_PATH = 'raw_data/biocatch_sinthetic_data.csv'
df_original = pd.read_csv(RAW_DATA_PATH)

# We will exclude features that the EDA recommended discarding or that are not interpolable (raw timestamps)
# We keep the numeric ones to apply SMOTE directly
exclude_cols = ['session_id', 'customer_id', 'session_timestamp', 'device_type', 'claim_category', 
                'biocatch_risk_score', 'biocatch_genuine_score', 'biocatch_ato_indicator', 
                'biocatch_social_eng_indicator', 'biocatch_bot_indicator','device_type','os_type', 'app_version']

target_col = 'is_vishing'

features = [c for c in df_original.columns if c not in exclude_cols and c != target_col]

X = df_original[features].fillna(0) # Simple fill n/a for processing
y = df_original[target_col]

majority_count = sum(y == 0)
minority_count = sum(y == 1)
total_count = len(y)

print(f"Total records: {total_count}")
print(f"Majority Class (Legitimate, 0): {majority_count} ({majority_count/total_count:.2%})")
print(f"Minority Class (Vishing,   1): {minority_count} ({minority_count/total_count:.2%})")

Total records: 50000
Majority Class (Legitimate, 0): 47500 (95.00%)
Minority Class (Vishing,   1): 2500 (5.00%)


## 3. Define General Pipeline Functions

In [3]:
def get_sampling_strategy(pct, majority_samples):
    """
    Computes the ratio for imblearn (minority / majority) given a target percentage 
    for the minority class in the full set.
    Equation: N_min = (pct / (1 - pct)) * N_maj
    """
    ratio = pct / (1.0 - pct)
    return ratio

def save_dataset(X_res, y_res, technique, pct, base_dir='data/balanced/original'):
    """
    Takes resampled X and y matrices, concatenates them, and exports to the file system.
    Expected format: data/technique/percentage.csv
    """
    dir_path = os.path.join(base_dir, technique)
    os.makedirs(dir_path, exist_ok=True)
    
    # Format the file name (e.g. 10.csv)
    pct_str = f"{int(pct * 100)}"
    file_path = os.path.join(dir_path, f"{pct_str}.csv")
    
    df_out = pd.concat([X_res.copy(), y_res.copy()], axis=1)
    df_out.to_csv(file_path, index=False)
    print(f"✅ Saved: {file_path} (Vishing distribution: {df_out['is_vishing'].mean():.2%}) | Total: {len(df_out)} rows")

TARGET_PERCENTAGES = [0.10, 0.20, 0.25]

## 4. Random Oversampling

In [4]:
print("Generating Random Oversampling...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    ros = RandomOverSampler(sampling_strategy=strategy, random_state=42)
    X_res, y_res = ros.fit_resample(X, y)
    save_dataset(X_res, y_res, 'random_oversampling', pct)

Generating Random Oversampling...
✅ Saved: data/balanced/original\random_oversampling\10.csv (Vishing distribution: 10.00%) | Total: 52777 rows
✅ Saved: data/balanced/original\random_oversampling\20.csv (Vishing distribution: 20.00%) | Total: 59375 rows
✅ Saved: data/balanced/original\random_oversampling\25.csv (Vishing distribution: 25.00%) | Total: 63333 rows


## 5. SMOTE

In [5]:
print("Generating SMOTE...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    smote = SMOTE(sampling_strategy=strategy, random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    save_dataset(X_res, y_res, 'smote', pct)

Generating SMOTE...
✅ Saved: data/balanced/original\smote\10.csv (Vishing distribution: 10.00%) | Total: 52777 rows
✅ Saved: data/balanced/original\smote\20.csv (Vishing distribution: 20.00%) | Total: 59375 rows
✅ Saved: data/balanced/original\smote\25.csv (Vishing distribution: 25.00%) | Total: 63333 rows


## 6. Borderline SMOTE

In [6]:
print("Generating Borderline SMOTE...")
for pct in TARGET_PERCENTAGES:
    strategy = get_sampling_strategy(pct, majority_count)
    bsmote = BorderlineSMOTE(sampling_strategy=strategy, random_state=42, kind='borderline-1')
    X_res, y_res = bsmote.fit_resample(X, y)
    save_dataset(X_res, y_res, 'borderline_smote', pct)

Generating Borderline SMOTE...
✅ Saved: data/balanced/original\borderline_smote\10.csv (Vishing distribution: 10.00%) | Total: 52777 rows
✅ Saved: data/balanced/original\borderline_smote\20.csv (Vishing distribution: 20.00%) | Total: 59375 rows
✅ Saved: data/balanced/original\borderline_smote\25.csv (Vishing distribution: 25.00%) | Total: 63333 rows


## 7. SMOTE + Undersampling (Pipeline)

In [7]:
print("Generating SMOTE + Undersampling...")
# Strategy: we raise Vishing drastically, and lower the majority class.
for pct in TARGET_PERCENTAGES:
    
    target_majority_count = int(majority_count * 0.9)  # Lower legitimate by 10%
    target_minority_count = int(target_majority_count * (pct / (1 - pct)))
    
    # Pipeline with SMOTE (oversample minority to target_minority_count) 
    # and RandomUnderSampler (undersample majority to target_majority_count)
    over = SMOTE(sampling_strategy={1: target_minority_count}, random_state=42)
    under = RandomUnderSampler(sampling_strategy={0: target_majority_count}, random_state=42)
    
    steps = [('o', over), ('u', under)]
    pipeline = Pipeline(steps=steps)
    
    X_res, y_res = pipeline.fit_resample(X, y)
    save_dataset(X_res, y_res, 'smote_undersampling', pct)

Generating SMOTE + Undersampling...
✅ Saved: data/balanced/original\smote_undersampling\10.csv (Vishing distribution: 10.00%) | Total: 47500 rows
✅ Saved: data/balanced/original\smote_undersampling\20.csv (Vishing distribution: 20.00%) | Total: 53437 rows
✅ Saved: data/balanced/original\smote_undersampling\25.csv (Vishing distribution: 25.00%) | Total: 57000 rows
